<a href="https://colab.research.google.com/github/Umesh-369/Language-Detection/blob/main/LanguageDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Language Detection using SVM (Colab / Jupyter version)
--------------------------------------------------------
Trains an SVM classifier (TF-IDF + LinearSVC) on the Language_Detection.csv
dataset and provides an interactive text box (ipywidgets) inside the
notebook where the user can type text and get the detected language.

Dataset expected columns: "Text", "Language"

Run this directly in a Colab / Jupyter cell.
"""

import re
import string
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import ipywidgets as widgets
from IPython.display import display, clear_output

# ----------------------------
# 1. Load Dataset
# ----------------------------
DATA_PATH = "/content/Language Detection.csv"  # update path if needed

df = pd.read_csv(DATA_PATH)
df.dropna(inplace=True)

# ----------------------------
# 2. Text Cleaning
# ----------------------------
def clean_text(text):
    text = str(text)
    text = re.sub(r'[!@#$(),\n"%^*?\:;~`0-9]', ' ', text)
    text = re.sub(r'\[.*?\]', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Text'] = df['Text'].apply(clean_text)

X = df['Text']
y = df['Language']

# ----------------------------
# 3. Train-Test Split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ----------------------------
# 4. TF-IDF Vectorization
# ----------------------------
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1, 3), max_features=20000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ----------------------------
# 5. Train SVM Model
# ----------------------------
model = LinearSVC()
model.fit(X_train_tfidf, y_train)

# ----------------------------
# 6. Evaluate Model
# ----------------------------
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ----------------------------
# 7. Prediction Function
# ----------------------------
def predict_language(text):
    text = clean_text(text)
    text_tfidf = vectorizer.transform([text])
    prediction = model.predict(text_tfidf)
    return prediction[0]

# ----------------------------
# 8. Interactive Widget UI (works in Colab / Jupyter)
# ----------------------------
input_box = widgets.Textarea(
    value='',
    placeholder='Type or paste text here...',
    description='Text:',
    layout=widgets.Layout(width='500px', height='120px')
)

detect_btn = widgets.Button(description="Detect Language", button_style='success')
clear_btn = widgets.Button(description="Clear", button_style='danger')
output = widgets.Output()

def on_detect_clicked(b):
    with output:
        clear_output()
        text = input_box.value.strip()
        if not text:
            print("⚠️  Please enter some text.")
            return
        result = predict_language(text)
        print(f"Detected Language: {result}")

def on_clear_clicked(b):
    input_box.value = ''
    with output:
        clear_output()

detect_btn.on_click(on_detect_clicked)
clear_btn.on_click(on_clear_clicked)

print("\nLanguage Detection (SVM) — enter text below and click 'Detect Language'\n")
display(input_box, widgets.HBox([detect_btn, clear_btn]), output)

Accuracy: 0.9864603481624759

Classification Report:
               precision    recall  f1-score   support

      Arabic       0.98      1.00      0.99       107
      Danish       0.98      0.97      0.97        86
       Dutch       0.97      0.95      0.96       109
     English       0.99      0.99      0.99       277
      French       0.99      0.99      0.99       203
      German       0.99      0.99      0.99        94
       Greek       1.00      1.00      1.00        73
       Hindi       1.00      1.00      1.00        12
     Italian       0.97      0.97      0.97       140
     Kannada       1.00      1.00      1.00        74
   Malayalam       1.00      1.00      1.00       119
  Portugeese       0.98      0.98      0.98       148
     Russian       1.00      0.99      0.99       138
     Spanish       0.98      0.99      0.98       164
    Sweedish       0.99      0.99      0.99       135
       Tamil       1.00      1.00      1.00        94
     Turkish       1.00    

Textarea(value='', description='Text:', layout=Layout(height='120px', width='500px'), placeholder='Type or pas…

Output()